In [1]:
import pandas as pd
import sys
from run_exp import run_experiment, Experiment
sys.path.insert(0, "..")

from src.features.features import (
    agg_crops, agg_surplus, agg_weather_w_lag, site_static, daily_nitrate,
    doy_climatology_pure_signal, nitrate_violations
)
from src.features.transformers import flatten_buckets, merge_on_date

# The two static-feature families site_static() emits on top of the base descriptors
# (lat/lon/log_basin_area/dist). Naming them lets each recipe drop the family it ablates.
COMID_KEYS  = ("cat_tiles92", "tot_tiles92", "cat_bfi", "tot_bfi", "cat_contact", "tot_contact")
AGTILE_KEYS = ("tile_frac_ag", "tile_frac_basin")

edges=(50_000,)
vel=2.1
lam=100_000

def recipe_A(site_uid):
    """Covariates + base static descriptors only -- NO COMID-keyed and NO AgTile features.
    The 'neither new family' baseline for the ablation."""
    e = list(edges)
    n = daily_nitrate(site_uid=site_uid)  # date spine; only .index is used
    df = merge_on_date(
        [
            nitrate_violations(site_uid=site_uid).rename("violation"),
            flatten_buckets(agg_weather_w_lag(site_uid=site_uid, edges=e, exp=False, water_velocity=vel)),
            flatten_buckets(agg_crops(site_uid=site_uid, edges=e, lam=lam, exp=True)),
            flatten_buckets(agg_surplus(site_uid=site_uid, edges=e, lam=lam, exp=True)),
            doy_climatology_pure_signal(n),
        ],
        spine=n.index,
    )
    drop = set(COMID_KEYS) | set(AGTILE_KEYS)
    for k, v in site_static(site_uid=site_uid).items():
        if k not in drop:
            df[k] = v
    return df


def recipe_B(site_uid):
    """recipe_B + everything but COMID-keyed features features"""
    e = list(edges)
    n = daily_nitrate(site_uid=site_uid)  # date spine; only .index is used
    df = merge_on_date(
        [
            nitrate_violations(site_uid=site_uid).rename("violation"),
            flatten_buckets(agg_weather_w_lag(site_uid=site_uid, edges=e, exp=False, water_velocity=vel)),
            flatten_buckets(agg_crops(site_uid=site_uid, edges=e, lam=lam, exp=True)),
            flatten_buckets(agg_surplus(site_uid=site_uid, edges=e, lam=lam, exp=True)),
            doy_climatology_pure_signal(n),
        ],
        spine=n.index,
    )
    for k, v in site_static(site_uid=site_uid).items():
        if k not in set(COMID_KEYS):
            df[k] = v
    return df

def recipe_C(site_uid):
    """recipe_B + everything but AgTiles features features"""
    e = list(edges)
    n = daily_nitrate(site_uid=site_uid)  # date spine; only .index is used
    df = merge_on_date(
        [ 
            nitrate_violations(site_uid=site_uid).rename("violation"),
            flatten_buckets(agg_weather_w_lag(site_uid=site_uid, edges=e, exp=False, water_velocity=vel)),
            flatten_buckets(agg_crops(site_uid=site_uid, edges=e, lam=lam, exp=True)),
            flatten_buckets(agg_surplus(site_uid=site_uid, edges=e, lam=lam, exp=True)),
            doy_climatology_pure_signal(n),
        ],
        spine=n.index,
    )
    for k, v in site_static(site_uid=site_uid).items():
        if k not in set(AGTILE_KEYS):
            df[k] = v
    return df

def recipe_D(site_uid):
    """recipe_A + COMID AND AgTiles"""
    e = list(edges)
    n = daily_nitrate(site_uid=site_uid)  # date spine; only .index is used
    df = merge_on_date(
        [
            nitrate_violations(site_uid=site_uid).rename("violation"),
            flatten_buckets(agg_weather_w_lag(site_uid=site_uid, edges=e, exp=False, water_velocity=vel)),
            flatten_buckets(agg_crops(site_uid=site_uid, edges=e, lam=lam, exp=True)),
            flatten_buckets(agg_surplus(site_uid=site_uid, edges=e, lam=lam, exp=True)),
            doy_climatology_pure_signal(n),
        ],
        spine=n.index,
    )
    for k, v in site_static(site_uid=site_uid).items():
        df[k] = v
        
    return df

recipes = {
    "base" : recipe_A,
    "base + AgTiles" : recipe_B,
    "base + COMID" : recipe_C,
    "base + both" : recipe_D
}

run_experiment(
    question="Do COMID and AgTiles improve the classifier results on a small feature set?",
    recipes=recipes,
    task="clf",
    mode="med",
    extra=True
)


======= exp adhoc [clf] : Do COMID and AgTiles improve the classifier results on a small feature set? =======
compare_many: [1/4] base                         elapsed    0s
  cooked base: 20/20 sites usable (min_rows >= 500)                    
  cooked base: 20 sites, 5-fold LOSO (~16 train / ~4 test per fold)
  extra_importance_test: permutation fold 5/5 (45 feats x 5 shuffles)
compare_many: [2/4] base + AgTiles               elapsed  371s
  cooked base + AgTiles: 20/20 sites usable (min_rows >= 500)                    
  cooked base + AgTiles: 20 sites, 5-fold LOSO (~16 train / ~4 test per fold)
  extra_importance_test: permutation fold 5/5 (47 feats x 5 shuffles)
compare_many: [3/4] base + COMID                 elapsed  696s
  cooked base + COMID: 20/20 sites usable (min_rows >= 500)                    
  cooked base + COMID: 20 sites, 5-fold LOSO (~16 train / ~4 test per fold)
  extra_importance_test: permutation fold 5/5 (51 feats x 5 shuffles)
compare_many: [4/4] base + both   

,n_sites,n_families,n_rows,n_feat,loso_auc,lofo_auc,loso_prauc,lofo_prauc,loso_f1,lofo_f1,...,loso_recall_at_far,lofo_prauc_lift,lofo_f2,lofo_mcc,lofo_recall_at_far,brier,persist_skill,base,between_rate_r2,macro_auc
recipe,,,,,,,,,,,,,,,,,,,,,
base,20,7,64924,45,0.801656,0.758093,0.486935,0.451007,0.541258,0.497900,...,0.424288,2.085555,0.635877,0.332331,0.380556,0.151466,-4.203055,0.216253,-0.891362,0.884128
base + AgTiles,20,7,64924,47,0.839984,0.828219,0.602027,0.584255,0.589942,0.576643,...,0.553846,2.701720,0.697396,0.444745,0.508120,0.126872,-3.355531,0.216253,0.026611,0.889662
base + COMID,20,7,64924,51,0.833972,0.816938,0.598145,0.575869,0.580776,0.561297,...,0.535256,2.662942,0.677844,0.425621,0.479416,0.128463,-3.410846,0.216253,0.046616,0.892398
base + both,20,7,64924,53,0.836966,0.825937,0.609776,0.597938,0.585530,0.573023,...,0.552991,2.764995,0.685707,0.442535,0.498575,0.126439,-3.339825,0.216253,0.108878,0.889007


In [2]:
run_experiment(
    question="Do COMID and AgTiles improve the classifier results on a small feature set?",
    recipes=recipes,
    task="clf",
    mode="full",
    extra=True
)


======= exp adhoc [clf] : Do COMID and AgTiles improve the classifier results on a small feature set? =======
compare_many: [1/4] base                         elapsed    0s
  cooked base: 45/45 sites usable (min_rows >= 500)                    
  cooked base: 45 sites, 5-fold LOSO (~36 train / ~9 test per fold)
  extra_importance_test: permutation fold 5/5 (45 feats x 5 shuffles)
compare_many: [2/4] base + AgTiles               elapsed  625s
  cooked base + AgTiles: 45/45 sites usable (min_rows >= 500)                    
  cooked base + AgTiles: 45 sites, 5-fold LOSO (~36 train / ~9 test per fold)
  extra_importance_test: permutation fold 5/5 (47 feats x 5 shuffles)
compare_many: [3/4] base + COMID                 elapsed 1353s
  cooked base + COMID: 45/45 sites usable (min_rows >= 500)                    
  cooked base + COMID: 45 sites, 5-fold LOSO (~36 train / ~9 test per fold)
  extra_importance_test: permutation fold 5/5 (51 feats x 5 shuffles)
compare_many: [4/4] base + both   

,n_sites,n_families,n_rows,n_feat,loso_auc,lofo_auc,loso_prauc,lofo_prauc,loso_f1,lofo_f1,...,loso_recall_at_far,lofo_prauc_lift,lofo_f2,lofo_mcc,lofo_recall_at_far,brier,persist_skill,base,between_rate_r2,macro_auc
recipe,,,,,,,,,,,,,,,,,,,,,
base,45,13,120004,45,0.814591,0.807423,0.601929,0.597387,0.619257,0.606345,...,0.476775,2.166547,0.723801,0.435230,0.462420,0.160658,-4.860479,0.275732,-0.032121,0.885515
base + AgTiles,45,13,120004,47,0.820885,0.817396,0.629623,0.590507,0.623880,0.616193,...,0.502463,2.141592,0.736540,0.451306,0.483121,0.154959,-4.651880,0.275732,0.047558,0.884181
base + COMID,45,13,120004,51,0.810285,0.816299,0.619340,0.599086,0.610091,0.622169,...,0.483152,2.172707,0.728541,0.458068,0.480552,0.160610,-4.859644,0.275732,-0.078306,0.883909
base + both,45,13,120004,53,0.809795,0.817660,0.622336,0.601387,0.615383,0.626407,...,0.488773,2.181054,0.730550,0.464306,0.486295,0.158800,-4.794193,0.275732,-0.084991,0.891131
